In [8]:
# =============================================================================
# Custom SLM — Kaggle 2x Tesla T4 (2x16 GB VRAM) — Train from Scratch
#
# Kaggle Settings (right-hand panel) before running:
#   Accelerator  → GPU T4 x2
#   Internet     → On
#
# Memory budget per GPU (16 GB each, total 32 GB):
#   Model weights  fp16   ~3 GB for 1.3 B params
#   Gradients      fp16   ~3 GB
#   8-bit AdamW           ~0.75 GB   (vs ~6 GB for fp32 Adam)
#   Activations  (w/ GC)  ~2–3 GB
#   ─────────────────────────────
#   Estimated total        ~9–10 GB / GPU  → safe on T4
#
# Techniques used (all from the project's own src/ package):
#   • DDP via accelerate.notebook_launcher     (2 GPUs, equal utilisation)
#   • bitsandbytes 8-bit AdamW                 (make_optimizer use_8bit=True)
#   • float16 AMP                              (T4 has fast fp16, no bf16)
#   • Gradient checkpointing                   (transformer use_checkpoint=True)
#   • Gradient accumulation × 16              (effective batch = 32)
#   • Depth-scaled residual init               (transformer._init_weights)
#   • RoPE with precomputed buffers            (MultiHeadAttention._apply_rope)
#   • SwiGLU feed-forward                      (FeedForward)
#   • Cosine LR schedule with linear warmup    (train.get_cosine_schedule_with_warmup)
#   • Unlikelihood loss                         (train._repetition_ul_loss)
#   • Step-based checkpointing                 (auto-resume across sessions)
# =============================================================================

## Cell 1 — Install dependencies
#
Run this cell once, then restart the kernel.

In [9]:
!pip install -q -U accelerate bitsandbytes datasets transformers huggingface_hub
# Install the project package from GitHub (or set REPO_ROOT below instead):
!pip install -q git+https://github.com/sh20022002/small-Language-Model.git

# Verify that key packages are at mutually compatible versions
import subprocess as _sp
_r = _sp.run(['pip', 'show', 'datasets', 'huggingface_hub', 'accelerate'],
             capture_output=True, text=True)
for _line in _r.stdout.split('\n'):
    if _line.startswith(('Name:', 'Version:')):
        print(_line)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Name: datasets
Version: 4.8.5
Name: huggingface_hub
Version: 1.16.1
Name: accelerate
Version: 1.13.0


## Cell 2 — sys.path setup + GPU + compute benchmark

In [10]:
import sys
from pathlib import Path

# ── If you uploaded the repo as a Kaggle dataset instead of pip-installing: ──
# Set REPO_ROOT to the dataset path, e.g. "/kaggle/input/small-language-model"
# Leave None when installed via pip.
REPO_ROOT = None    # e.g.  "/kaggle/input/small-language-model"

if REPO_ROOT:
    for sub in ("src", "tests"):
        p = str(Path(REPO_ROOT) / sub)
        if p not in sys.path:
            sys.path.insert(0, p)

# ── GPU info via nvidia-smi (no torch.cuda — keeps CUDA uninitialised so
#    notebook_launcher can use fork/spawn without hitting the
#    "Cannot re-initialize CUDA in forked subprocess" error) ──────────────────
import subprocess

try:
    _smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    )
    _gpu_lines = [l.strip() for l in _smi.stdout.strip().split("\n") if l.strip()]
    n_gpus = len(_gpu_lines)
    print(f"GPUs available: {n_gpus}")
    for i, line in enumerate(_gpu_lines):
        name, mem = line.split(",")
        print(f"  cuda:{i}  {name.strip()}  {float(mem.split()[0]) / 1024:.1f} GB")
except Exception as _e:
    print(f"[nvidia-smi] {_e}")
    n_gpus = 0

if n_gpus < 2:
    print("\nWARNING: fewer than 2 GPUs found — "
          "enable 'GPU T4 x2' in Kaggle Settings → Accelerator.")

# Compute benchmark — uses tests/mfu.py from the project
try:
    from mfu import run_perf_and_mfu
    if n_gpus >= 1:
        run_perf_and_mfu()
except ImportError:
    print("[mfu] Skipped — set REPO_ROOT above to enable, or pip-install the package.")

GPUs available: 2
  cuda:0  Tesla T4  15.0 GB
  cuda:1  Tesla T4  15.0 GB
[mfu] Skipped — set REPO_ROOT above to enable, or pip-install the package.


## Cell 3 — Configuration
**Edit this cell only.** Everything else adapts automatically.

In [11]:
# ── Model configuration ────────────────────────────────────────────────────────
# Estimated parameter counts (GPT-2 tokenizer, vocab=50 257):
#   tiny   : dim=256,  depth=4,  heads=4,  mlp=1024   → ~30 M   (quick smoke test)
#   small  : dim=512,  depth=8,  heads=8,  mlp=2048   → ~110 M
#   medium : dim=768,  depth=12, heads=12, mlp=3072   → ~250 M
#   large  : dim=1024, depth=16, heads=16, mlp=4096   → ~650 M
#   xlarge : dim=1536, depth=24, heads=24, mlp=6144   → ~1.3 B  ← default
MODEL_CFG = dict(
    dim     = 512,
    depth   = 8,
    heads   = 8,
    mlp_dim = 2048,
    window  = 2048,     # local-attention window = max sequence length
    dropout = 0.1,
)

# ── Tokenizer ──────────────────────────────────────────────────────────────────
# A) Load a saved HybridTokenizer (from a previous Colab run):
#      TOKENIZER_PATH = "/kaggle/input/my-tokenizer/tokenizer.pkl.gz"
# B) Build HybridTokenizer on-the-fly from TinyStories (~2 min):
#      BUILD_HYBRID = True
# C) Default — GPT-2 BPE (50 257 tokens, no auth, instant):
TOKENIZER_PATH = None       # str path to .pkl.gz, or None
BUILD_HYBRID   = False      # True = build from TinyStories at startup
HF_TOKENIZER   = "gpt2"    # used only when A and B are both disabled

# ── Training stages (curriculum order) ────────────────────────────────────────
# [dataset_name, steps]  —  names: tinystories | wikitext | openwebtext | alpaca
STAGES = [
    ["tinystories",  2000],
    ["wikitext",     3000],
    ["openwebtext",  3000],
    ["alpaca",       1000],
]
MAX_ITEMS_TRAIN = 50_000
MAX_ITEMS_VAL   =  2_000

# ── Training hyper-parameters ──────────────────────────────────────────────────
BATCH_SIZE    = 1       # per GPU; effective = 1 × 2 GPUs × 16 accum = 32
GRAD_ACCUM    = 16
LR            = 3e-4
WEIGHT_DECAY  = 0.1
MAX_GRAD_NORM = 1.0
WARMUP_STEPS  = 200
UL_ALPHA      = 0.1     # unlikelihood-loss coefficient

# ── Checkpointing ──────────────────────────────────────────────────────────────
OUTPUT_DIR       = "/kaggle/working/slm_run"
SAVE_STEPS       = 200   # save every N *optimizer* steps
SAVE_TOTAL_LIMIT = 2     # keep only the last 2 checkpoints

## Cell 4 — Training function  (executed once per GPU via DDP)

In [12]:
def _train_fn():
    """
    Launched by notebook_launcher with num_processes=2.
    Each GPU process runs this function independently.
    All imports are local to avoid multiprocessing pickling issues.
    """
    import math, shutil
    from pathlib import Path

    import torch
    from accelerate import Accelerator
    from accelerate.utils import set_seed

    # ── Project imports ───────────────────────────────────────────────────────
    from my_slm.transformer import Transformer
    from my_slm.train import make_optimizer, get_cosine_schedule_with_warmup
    from my_slm.multi_train_orchestrator import StageConfig, train_across_datasets
    from my_slm.hybrid_tokeniztion import HybridTokenizer

    # ── Accelerator ───────────────────────────────────────────────────────────
    accelerator = Accelerator(
        mixed_precision            = "fp16",   # T4: fp16 only (no bf16)
        gradient_accumulation_steps= GRAD_ACCUM,
    )
    set_seed(42)
    is_main = accelerator.is_main_process

    if is_main:
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        print(f"[Accelerator] world={accelerator.num_processes}  "
              f"device={accelerator.device}  fp16=True  grad_accum={GRAD_ACCUM}")

    # ── Tokenizer ─────────────────────────────────────────────────────────────
    if TOKENIZER_PATH and Path(TOKENIZER_PATH).exists():
        tokenizer = HybridTokenizer.load(TOKENIZER_PATH)
        if is_main:
            print(f"[Tokenizer] HybridTokenizer loaded — vocab={tokenizer.vocab_size}")

    elif BUILD_HYBRID:
        from datasets import load_dataset
        tokenizer = HybridTokenizer()
        if is_main:
            print("[Tokenizer] Building HybridTokenizer from TinyStories …")
            stream = load_dataset("roneneldan/TinyStories",
                                  split="train", streaming=True)
            for i, ex in enumerate(stream):
                tokenizer.add_text(ex.get("text", ""))
                if i >= 20_000:
                    break
            tokenizer.freeze_vocab(k_bases=2000, max_merges=20_000)
            tok_out = Path(OUTPUT_DIR) / "tokenizer.pkl.gz"
            tokenizer.save(tok_out)
            print(f"[Tokenizer] Built — vocab={tokenizer.vocab_size}  → {tok_out}")
        accelerator.wait_for_everyone()
        if not is_main:
            tokenizer = HybridTokenizer.load(Path(OUTPUT_DIR) / "tokenizer.pkl.gz")

    else:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        if is_main:
            print(f"[Tokenizer] GPT-2 BPE — vocab={len(tokenizer)}")

    vocab_size = (tokenizer.vocab_size
                  if hasattr(tokenizer, "vocab_size") else len(tokenizer))

    # ── Model ─────────────────────────────────────────────────────────────────
    model = Transformer(
        vocab_size     = vocab_size,
        use_checkpoint = True,          # gradient checkpointing enabled
        **MODEL_CFG,
    )
    if is_main:
        n = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"[Model] {n/1e6:.1f} M params ({n/1e9:.3f} B)  "
              f"dim={MODEL_CFG['dim']}  depth={MODEL_CFG['depth']}  "
              f"heads={MODEL_CFG['heads']}")

    # ── Architecture sanity check (pre-training, main process only) ───────────
    if is_main:
        try:
            # tests/test_model.py — notebook-callable entry point
            from test_model import check_model_architecture
            ok = check_model_architecture(model, vocab_size, device="cpu")
            if not ok:
                raise RuntimeError("Architecture checks failed — fix before training!")
        except ImportError:
            print("[check] test_model.py not found — set REPO_ROOT to enable.")

    # ── Optimizer (8-bit AdamW from bitsandbytes) ─────────────────────────────
    optimizer = make_optimizer(
        model,
        lr           = LR,
        weight_decay = WEIGHT_DECAY,
        betas        = (0.9, 0.95),
        use_8bit     = True,            # ~8× smaller optimizer state on GPU
    )

    # ── LR scheduler ─────────────────────────────────────────────────────────
    total_opt_steps = sum(
        math.ceil(steps / GRAD_ACCUM) for _, steps in STAGES
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        warmup_steps = WARMUP_STEPS,
        total_steps  = total_opt_steps,
    )

    # ── Prepare with Accelerator ──────────────────────────────────────────────
    model, optimizer, scheduler = accelerator.prepare(model, optimizer, scheduler)

    # ── Checkpoint helpers ────────────────────────────────────────────────────
    def _save(step: int):
        if not is_main:
            return
        ckpt_dir = Path(OUTPUT_DIR) / f"checkpoint-{step}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        torch.save({
            "config": {**MODEL_CFG, "vocab_size": vocab_size},
            "model_state": accelerator.unwrap_model(model).state_dict(),
            "optimizer":   optimizer.state_dict(),
            "step":        step,
        }, ckpt_dir / "state.pt")
        # Enforce SAVE_TOTAL_LIMIT
        ckpts = sorted(
            [d for d in Path(OUTPUT_DIR).iterdir()
             if d.is_dir() and d.name.startswith("checkpoint-")],
            key=lambda d: int(d.name.split("-")[1]),
        )
        for old in ckpts[:-SAVE_TOTAL_LIMIT]:
            shutil.rmtree(old)
        print(f"[Checkpoint] step={step}  → {ckpt_dir}")

    def _resume() -> int:
        ckpts = sorted(
            [d for d in Path(OUTPUT_DIR).iterdir()
             if d.is_dir() and d.name.startswith("checkpoint-")],
            key=lambda d: int(d.name.split("-")[1]),
        ) if Path(OUTPUT_DIR).is_dir() else []
        if not ckpts:
            return 0
        state = torch.load(ckpts[-1] / "state.pt", map_location="cpu")
        accelerator.unwrap_model(model).load_state_dict(state["model_state"],
                                                        strict=False)
        optimizer.load_state_dict(state["optimizer"])
        if is_main:
            print(f"[Checkpoint] Resumed from step {state['step']}  ({ckpts[-1].name})")
        return state["step"]

    global_step = _resume()

    # ── Multi-stage curriculum training (via orchestrator) ────────────────────
    stages = [StageConfig(name, steps=steps) for name, steps in STAGES]

    model = train_across_datasets(
        model              = model,
        optimizer          = optimizer,
        tokenizer          = tokenizer,
        accelerator        = accelerator,
        stages             = stages,
        max_len            = MODEL_CFG["window"],
        train_items        = MAX_ITEMS_TRAIN,
        val_items          = MAX_ITEMS_VAL,
        batch_size         = BATCH_SIZE,
        scheduler          = scheduler,
        max_grad_norm      = MAX_GRAD_NORM,
        ul_alpha           = UL_ALPHA,
        save_dir           = OUTPUT_DIR,
    )

    global_step += sum(math.ceil(steps / GRAD_ACCUM) for _, steps in STAGES)
    _save(global_step)

    # ── Final weights ─────────────────────────────────────────────────────────
    accelerator.wait_for_everyone()
    if is_main:
        final_path = Path(OUTPUT_DIR) / "final_model.pt"
        torch.save({
            "config": {**MODEL_CFG, "vocab_size": vocab_size},
            "model_state": accelerator.unwrap_model(model).state_dict(),
        }, final_path)
        print(f"\n[Done] Final weights → {final_path}")

    accelerator.end_training()

## Cell 5 — Launch DDP (2 processes, one per T4)

In [25]:
# torchrun uses spawn (fresh processes) — no CUDA fork issues.
# sys.executable ensures the worker uses the SAME Python / site-packages
# as this Jupyter kernel, so pip-installed packages are always visible.
import inspect, json, os, subprocess, sys
from pathlib import Path

# ── Serialise all notebook config vars to disk ─────────────────────────
Path('/tmp/slm_cfg.json').write_text(json.dumps(dict(
    MODEL_CFG=MODEL_CFG, STAGES=STAGES,
    MAX_ITEMS_TRAIN=MAX_ITEMS_TRAIN, MAX_ITEMS_VAL=MAX_ITEMS_VAL,
    BATCH_SIZE=BATCH_SIZE, GRAD_ACCUM=GRAD_ACCUM, LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY, MAX_GRAD_NORM=MAX_GRAD_NORM,
    WARMUP_STEPS=WARMUP_STEPS, UL_ALPHA=UL_ALPHA,
    OUTPUT_DIR=OUTPUT_DIR, SAVE_STEPS=SAVE_STEPS,
    SAVE_TOTAL_LIMIT=SAVE_TOTAL_LIMIT,
    TOKENIZER_PATH=TOKENIZER_PATH, BUILD_HYBRID=BUILD_HYBRID,
    HF_TOKENIZER=HF_TOKENIZER, REPO_ROOT=REPO_ROOT,
)))

# ── Build worker script ───────────────────────────────────────────────────────────────────────
_preamble = [
    'import json, sys',
    'from pathlib import Path',
    'cfg = json.loads(Path("/tmp/slm_cfg.json").read_text())',
    'for k, v in cfg.items(): globals()[k] = v',
    'if globals().get("REPO_ROOT"):',
    '    for s in ("src", "tests"):',
    '        p = str(Path(globals()["REPO_ROOT"]) / s)',
    '        if p not in sys.path: sys.path.insert(0, p)',
    '']
_fn_src = inspect.getsource(_train_fn)
_worker_src = '\n'.join(_preamble) + _fn_src + '\nif __name__ == "__main__": _train_fn()\n'
Path('/tmp/slm_worker.py').write_text(_worker_src)

# ── Verify worker script ───────────────────────────────────────────────────────────────────────
print("=== Worker script (first 20 lines) ===")
for _i, _ln in enumerate(_worker_src.split('\n')[:20]):
    print(f"{_i+1:3d}  {_ln}")
print()

_chk = subprocess.run([sys.executable, '-m', 'py_compile', '/tmp/slm_worker.py'],capture_output=True, text=True)
if _chk.returncode != 0:
    raise SyntaxError(f"Worker script syntax error:{_chk.stderr}")
print("[OK] Syntax check passed")
# ── Launch via torch.distributed.run (same Python as this kernel) ──────────────────
# Using sys.executable guarantees the worker processes inherit the exact same
# Python interpreter and site-packages as this Jupyter kernel, so every
# pip-installed package (datasets, huggingface_hub, my_slm, ...) is available.
_cmd = (
    f'{sys.executable} -m torch.distributed.run'
    f' --standalone --nproc_per_node={n_gpus or 2}'
    ' --master_port=29500'
    ' /tmp/slm_worker.py')
print(f"Running: {_cmd}")
_ret = os.system(_cmd)
if _ret != 0:
    raise RuntimeError(f'torchrun exited with code {_ret}')

=== Worker script (first 20 lines) ===


ValueError: empty separator

## Cell 6 — Post-training checks + semantic evaluation
#
Run after Cell 5 completes.  Requires REPO_ROOT set in Cell 2.

In [ ]:
def _post_training_eval(quick: bool = True):
    """
    1. Architecture + training behaviour checks (test_model, test_training).
    2. Semantic benchmarks (perplexity, top-k, BLiMP, LAMBADA, analogy).
    """
    from pathlib import Path
    import torch

    final_path = Path(OUTPUT_DIR) / "final_model.pt"
    if not final_path.exists():
        print(f"[Eval] {final_path} not found — run Cell 5 first.")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load model + tokenizer via semantic_eval helper
    try:
        from semantic_eval import load_model_and_tok, run_all, print_report
    except ImportError:
        print("[Eval] semantic_eval.py not found — set REPO_ROOT in Cell 2.")
        return

    tok_src = TOKENIZER_PATH or HF_TOKENIZER
    model, tok = load_model_and_tok(str(final_path), tok_src, device)

    # ── Architecture checks ───────────────────────────────────────────────────
    try:
        from test_model import check_model_architecture
        check_model_architecture(model, model.token_emb.num_embeddings, device)
    except ImportError:
        pass

    # ── Post-training behaviour checks ────────────────────────────────────────
    try:
        from test_training import check_trained_model
        from my_slm.multi_train_orchestrator import _get_pad_id
        check_trained_model(model, tok, device,
                            vocab_size=model.token_emb.num_embeddings,
                            pad_id=_get_pad_id(tok))
    except ImportError:
        pass

    # ── Semantic benchmarks ───────────────────────────────────────────────────
    report = run_all(model, tok, device, quick=quick)
    print_report(report)
    return report


# Uncomment to run:
# _post_training_eval(quick=True)

## Cell 7 — Generation test

In [ ]:
def _generate(prompt: str, max_new_tokens: int = 120, temperature: float = 0.8):
    from pathlib import Path
    import torch
    from my_slm.transformer import Transformer
    from my_slm.multi_train_orchestrator import _encode, _get_pad_id

    final_path = Path(OUTPUT_DIR) / "final_model.pt"
    ckpt = torch.load(final_path, map_location="cpu")

    cfg        = ckpt["config"]
    state_dict = ckpt["model_state"]

    if TOKENIZER_PATH and Path(TOKENIZER_PATH).exists():
        from my_slm.hybrid_tokeniztion import HybridTokenizer
        tok = HybridTokenizer.load(TOKENIZER_PATH)
    else:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(HF_TOKENIZER)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token

    model = Transformer(**{k: cfg[k] for k in
                           ("vocab_size", "dim", "depth", "heads", "mlp_dim", "window")},
                        dropout=0.0, use_checkpoint=False)
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    eos_id = _get_pad_id(tok)
    ids    = torch.tensor([_encode(tok, prompt)], dtype=torch.long)

    with torch.inference_mode():
        out = model.generate(
            ids,
            max_new_tokens   = max_new_tokens,
            temperature      = temperature,
            top_k            = 50,
            suppress_ids     = [eos_id],
            repetition_penalty = 1.3,
        )

    gen_ids = out[0, len(_encode(tok, prompt)):].tolist()
    if hasattr(tok, "token2id"):
        return tok.decode(gen_ids)
    return tok.decode(gen_ids, skip_special_tokens=True)


# print(_generate("Once upon a time"))   # uncomment to test